# Tome 13, Chapter 3: Sieves and the parity problem (Python companion)

Exercises first, solutions at the end. We watch Brun's truncated inclusion–exclusion oscillate around the truth, test the twin prime bounds of the chapter, look at Selberg's parity example in actual numbers, and measure Artin's densities. The computer science application: **how to find a generator for Diffie–Hellman**, where the structure of $p-1$ decides how easy the test is, and where Artin's constant reappears. Requires `numpy` and `sympy`.

In [ ]:
import math, random, time
import numpy as np
from sympy import primerange, isprime, randprime, factorint
random.seed(13)

def spf_sieve(N):
    spf = np.zeros(N + 1, dtype=np.int32)
    for i in range(2, int(N**0.5) + 1):
        if spf[i] == 0:
            block = spf[i*i::i]; block[block == 0] = i; spf[i*i::i] = block
    return spf

N = 10**6
spf = spf_sieve(N)
is_prime = (spf == 0); is_prime[:2] = False
primes = np.nonzero(is_prime)[0]

## Exercises

**P3.1 (★) Brun's truncations.** For $x=10^6$ and $z=30$ ($P(z)=2\cdot3\cdot5\cdots29$), compute the truncated Legendre sums $\sum_{d\mid P(z),\,\omega(d)\le r}\mu(d)\lfloor x/d\rfloor$ for $r=0,1,\dots,10$, and compare with the true count $\Phi(x,z)$ of $n\le x$ free of primes up to $z$. Check the alternating signs of Lemma 3.1. Then do the same for $z=100$: where are the errors larger, and why do both computations become exact at the same $r$?

**P3.2 (★★) Twin prime bounds.** (a) Count $\pi_2(x)$ for $x=10^6$ and compare with $2C_2\frac x{\log^2x}$, with $2C_2\int_2^x\frac{dt}{\log^2t}$, and with the sieve bound $16C_2\frac x{\log^2x}$ (Exercise 3.6). (b) Compute Selberg's $G(y)=\sum_{d\le y}\mu^2(d)h(d)$, with $h(2)=1$ and $h(p)=\frac2{p-2}$, for $y=10^2,\dots,10^6$, and compare with $\frac{\log^2y}{4C_2}$. How large is the $O(\log y)$ term?

**P3.3 (★★) Parity in numbers.** For $x=10^6$, let $\mathcal A^\pm$ be the $n\le x$ with $\lambda(n)=\pm1$. (a) Compare $|\mathcal A^+_d|$ and $|\mathcal A^-_d|$ with $\frac x{2d}$ for $d=1,\dots,10$ and a few larger $d$. (b) Compute $S(\mathcal A^\pm,x^{1/2})$ and $S(\mathcal A^\pm,x^{1/3})$, and identify the survivors (Exercise 3.5).

**P3.4 (★★) CS application: Diffie–Hellman generators.** Diffie–Hellman key exchange works in $(\mathbb Z/p\mathbb Z)^\times$ with a generator $g$ (or of a large subgroup). To check that $g$ is a primitive root one needs the prime factors $q$ of $p-1$ and tests $g^{(p-1)/q}\ne1$ (Lemma 3.14).

(a) For each squarefree $a\in\{2,3,5,6,7,10,11,13\}$, compute the proportion of primes $p\le10^6$ modulo which $a$ is a primitive root, and compare with Hooley's density: $A$ if $a\not\equiv1\pmod4$, and $A\bigl(1-\mu(a)\prod_{q\mid a}\frac1{q^2-q-1}\bigr)$ if $a\equiv1\pmod4$.

(b) For a *safe prime* $p=2q+1$ ($q$ prime), $g$ is a primitive root iff $g^2\ne1$ and $g^q\ne1$. Generate a $256$-bit safe prime, and for $g=2,3,4,\dots$ find the first primitive root. What proportion of all residues are primitive roots, and why is it about $\frac12$ here but about $A$ for a random prime?

(c) Average $\frac{\varphi(p-1)}{p-1}$ over the primes $p\le10^6$, and compare with $A$. Explain why the average density of primitive roots among residues is Artin's constant, while for a single $a$ the density of primes is Artin's constant: two sides of the same double count.

## Solutions

### P3.1
**How to find it.** Enumerate the squarefree divisors of $P(z)$ by subsets of the primes (there are $2^{10}$ for $z=30$ and $2^{25}$ for $z=100$, too many to list: group them by $\omega(d)$ and stop at $r$, using a recursive enumeration that only goes $r$ deep).

In [ ]:
def truncated(x, plist, r):
    # sum over squarefree d | prod(plist) with omega(d) <= r of mu(d) * floor(x/d)
    total = 0
    def rec(start, d, k):
        nonlocal total
        total += (-1)**k * (x // d)
        if k == r:
            return
        for i in range(start, len(plist)):
            nd = d * plist[i]
            if nd > x:
                break
            rec(i + 1, nd, k + 1)
    rec(0, 1, 0)
    return total

def Phi(x, z):
    alive = np.ones(x + 1, bool); alive[0] = False
    for p in primerange(2, z + 1):
        alive[p::p] = False
    return int(alive.sum())

x = 10**6
for z in (30, 100):
    pl = list(primerange(2, z + 1))
    true = Phi(x, z)
    print(f"z={z}: Phi = {true}")
    for r in range(0, 11):
        t = truncated(x, pl, r)
        print(f"  r={r:2d}: {t:9d}  error {t-true:+9d}")

**What it shows.** For $z=30$ ($10$ primes) the errors alternate in sign, positive for even $r$ and negative for odd $r$, as Lemma 3.1 says. For $z=30$ they shrink from $842\,061$ at $r=0$ to $7348$ at $r=4$ and $7$ at $r=6$. For $z=100$ ($25$ primes) the middle errors are two to three times larger ($-151\,504$ against $-56\,100$ at $r=3$), because $G=\sum_{p\le100}\frac1p\approx1.80$ exceeds $\sum_{p\le30}\frac1p\approx1.38$ and the terms $\frac{G^r}{r!}$ are larger. Both become exact at $r=7$ for a reason that has nothing to do with the sieve: a product of eight distinct primes is at least $2\cdot3\cdots19=9\,699\,690>x$, so $\lfloor x/d\rfloor=0$ for those $d$. For realistic sizes ($x$ huge and $z$ a power of $x$) this never happens, and Brun's cure in Theorem 3.4 is to truncate the small primes much later than the large ones.

### P3.2
**How to find it.** $\pi_2$ from the prime list. For $G(y)$, enumerate the squarefree $d\le y$ with a sieve of $\mu$, and compute $h(d)$ multiplicatively from the smallest prime factor.

In [ ]:
C2 = 1.0
for p in primerange(3, 10**6):
    C2 *= 1 - 1/(p-1)**2
x = 10**6
ps = set(primes.tolist())
pi2 = sum(1 for p in primes if p + 2 <= x and (p + 2) in ps)
li2 = sum(1/math.log(t)**2 for t in range(2, x + 1))
print(f"C2 = {C2:.7f}")
print(f"pi_2(1e6) = {pi2}; 2C2 x/log^2 x = {2*C2*x/math.log(x)**2:.0f}; "
      f"2C2 int = {2*C2*li2:.0f}; 16C2 x/log^2 x = {16*C2*x/math.log(x)**2:.0f}")

Y = 10**6
h = np.zeros(Y + 1); h[1] = 1.0
for n in range(2, Y + 1):
    p = spf[n] if spf[n] else n
    m = n // p
    if m % p == 0:
        h[n] = 0.0
    else:
        h[n] = h[m] * (1.0 if p == 2 else 2/(p - 2))
G = np.cumsum(h)
for y in (10**2, 10**3, 10**4, 10**5, 10**6):
    pred = math.log(y)**2/(4*C2)
    print(f"y={y:>7}: G={G[y]:9.2f}  log^2y/(4C2)={pred:9.2f}  diff/log y={(G[y]-pred)/math.log(y):6.3f}")

**What it shows.** $C_2=0.66016$, and $\pi_2(10^6)=8169$, against $2C_2\frac x{\log^2x}\approx6917$ and $2C_2\int_2^x\frac{dt}{\log^2t}\approx8250$: the integral is the right main term, as in Tome 5's Figure 8.1. The sieve bound $16C_2\frac x{\log^2x}\approx55\,340$ is about $6.8$ times the truth here. $G(y)$ exceeds $\frac{\log^2y}{4C_2}$ by about $2.3\log y$ over the whole range, a stable ratio: the $O(\log y)$ term of Exercise 3.6 is a genuine linear term $c\log y$, not noise, and it makes the sieve bound better than its limit at any finite $x$.

### P3.3
**How to find it.** $\lambda(n)=(-1)^{\Omega(n)}$ from the smallest-prime-factor table. Then $|\mathcal A^\pm_d|$ is a count over multiples of $d$.

In [ ]:
Omega = np.zeros(N + 1, dtype=np.int8)
for n in range(2, N + 1):
    p = spf[n] if spf[n] else n
    Omega[n] = Omega[n // p] + 1
lam = np.where(Omega % 2 == 0, 1, -1); lam[0] = 0
x = N
for d in list(range(1, 11)) + [101, 1009, 10007]:
    m = lam[d::d]
    plus, minus = int((m == 1).sum()), int((m == -1).sum())
    print(f"d={d:6d}: |A+_d|={plus:7d}  |A-_d|={minus:7d}  x/2d={x/(2*d):10.1f}")
for z, name in ((int(x**0.5), "x^(1/2)"), (int(round(x**(1/3))), "x^(1/3)")):
    alive = np.ones(x + 1, bool); alive[0] = False
    for p in primerange(2, z + 1):
        alive[p::p] = False
    surv = np.nonzero(alive)[0]
    sp = int((lam[surv] == 1).sum()); sm = int((lam[surv] == -1).sum())
    two = int(((Omega[surv] == 2)).sum())
    print(f"z={name}: S(A+)={sp} (of which {two} have two prime factors), S(A-)={sm}")

**What it shows.** The two lists have $|\mathcal A^\pm_d|$ within a fraction of a percent of $\frac x{2d}$ for every $d$ shown: at $d=1$ there are $499\,735$ (even) against $500\,265$ (odd), and the deviations from $\frac x{2d}$ are $\frac12|L(\frac xd)|$, tiny compared with $\frac x{2d}$. Yet at $z=\sqrt x$ only $n=1$ survives in $\mathcal A^+$, while $\mathcal A^-$ keeps the $78\,330$ primes between $1000$ and $10^6$. At $z=x^{1/3}=100$ both lists have many survivors: $\mathcal A^-$ the $78\,473$ primes above $100$, $\mathcal A^+$ the number $1$ and $42\,286$ products of two primes. Their ratio, $0.54$, tends to $\log2\approx0.69$ only slowly: the correction terms, such as the $\pi(p_1)$ pairs with $p_2<p_1$ excluded for each $p_1$, are of relative size $\frac1{\log x}$ (Exercise 3.5).

### P3.4
**How to find it.** (a) Factor $p-1$ with the smallest-prime-factor table and test $a^{(p-1)/q}\ne1$. (b) `randprime` for $q$ until $2q+1$ is prime. (c) $\varphi(p-1)$ from the factorisation.

In [ ]:
A = 1.0
for q in primerange(2, 10**6):
    A *= 1 - 1/(q*(q - 1))

def prime_factors(n):
    fs = []
    while n > 1:
        p = int(spf[n]) if spf[n] else n
        fs.append(p)
        while n % p == 0:
            n //= p
    return fs

def mu(n):
    f = factorint(n)
    return 0 if any(e > 1 for e in f.values()) else (-1)**len(f)

counts = {a: 0 for a in (2, 3, 5, 6, 7, 10, 11, 13)}
phi_ratio = 0.0
odd_primes = [int(p) for p in primes if p > 2]
for p in odd_primes:
    qs = prime_factors(p - 1)
    for a in counts:
        if a % p and all(pow(a, (p - 1)//q, p) != 1 for q in qs):
            counts[a] += 1
    r = 1.0
    for q in qs:
        r *= 1 - 1/q
    phi_ratio += r
P = len(odd_primes)
for a, c in counts.items():
    pred = A
    if a % 4 == 1:
        prod = 1.0
        for q in factorint(a):
            prod *= 1/(q*q - q - 1)
        pred = A*(1 - mu(a)*prod)
    print(f"a={a:2d}: observed {c/P:.4f}  Hooley {pred:.4f}")
print(f"average phi(p-1)/(p-1) over p <= 1e6: {phi_ratio/P:.4f}   A = {A:.7f}")

t0 = time.time()
tries = 0
while True:
    q = randprime(2**254, 2**255); tries += 1
    if isprime(2*q + 1):
        break
p = 2*q + 1
g = 2
while pow(g, 2, p) == 1 or pow(g, q, p) == 1:
    g += 1
print(f"256-bit safe prime found after {tries} candidates q ({time.time()-t0:.1f}s); first primitive root g = {g}")
small_safe = [int(2*q + 1) for q in primerange(3, 20000) if isprime(2*q + 1)]
frac = np.mean([(q - 1)/(2*q) for q in ((s - 1)//2 for s in small_safe)])
print(f"proportion of residues that are primitive roots, safe primes < 40000: {frac:.4f}")

**What it shows.** (a) Up to $10^6$ the proportions are within $0.002$ of Hooley's densities: $A=0.3740$ for $a=2,3,6,7,10,11$, $\frac{20}{19}A=0.3936$ for $a=5$, and $\frac{156}{155}A=0.3764$ for $a=13$ (the correction factor $1+\frac1{155}$ is too small to see at this height). (b) Here the search took $45$ random primes $q$ before $2q+1$ was prime, and the first primitive root was $g=13$; both numbers change with the random seed. That infinitely many safe primes exist is a problem of the same kind as twin primes, out of reach of sieves by parity (Exercise 3.1). For a safe prime, $\varphi(p-1)=q-1$, so almost exactly half of the residues are primitive roots, and the test needs only two exponentiations. That is why protocols use safe primes: the factorisation of $p-1$ is known, and the test is cheap. (c) The average of $\frac{\varphi(p-1)}{p-1}$ is about $0.374$, Artin's constant again: counting pairs $(a,p)$ with $a$ a primitive root modulo $p$ in two orders gives the same constant, on average over $a$ (Stephens, 1969) and, under GRH, for each fixed $a$ (Hooley).

*References.* W. Diffie and M. Hellman, "New directions in cryptography", *IEEE Trans. Inform. Theory* 22 (1976), 644–654. P. J. Stephens, "An average result for Artin's conjecture", *Mathematika* 16 (1969), 178–188. C. Hooley, "On Artin's conjecture", *J. Reine Angew. Math.* 225 (1967), 209–220.